In [ ]:
import os
import re
from typing import List
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# https://github.com/ranaroussi/quantstats
%matplotlib inline
import quantstats as qs

TIMEZONE : str = "Asia/Hong_Kong"
ECONOMIC_CALENDAR_FILE = r"economic_calanedar_archive.csv"

TICKER : str = "BTC"
CANDLES_CACHE_DIR : str = './cache'
REQUIRED_TA : List[str] = [ 'close', 'atr', 'adx' ]
RELOAD_CANDLES : bool = False # You dont want to refresh candles everytime you reload Equity Curve with diff filtering?

MAXPAIN_CUTOFF_BPS = 70
RISK_FREE_RATE = 0.035
ACC_EQUITY = 100000

RETURN_CALC_METHODOLOGY = "log" # log or simple
RETURN_RESAMPLE_FREQ = "D" # D for daily, W for weekly

In [ ]:
# backtest_core will spits out trade extract
trade_files = ['1_trades.csv', '2_trades.csv', '3_trades.csv']

In [ ]:
def candles_ta_files_2_dataframe(
    candles_ta_files : List[str]
):
    _candles_ta_files = []
    for candles_ta_file in candles_ta_files:
        pd_candles_ta = pd.read_csv(candles_ta_file)

        if not all(col in pd_candles_ta.columns for col in REQUIRED_TA):
            continue

        _candles_ta_files.append(pd_candles_ta)
    pd_candles_ta = pd.concat(_candles_ta_files, ignore_index=True)
    pd_candles_ta.drop_duplicates(subset='timestamp_ms', inplace=True)
    pd_candles_ta.drop(pd_candles_ta.columns[pd_candles_ta.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)
    pd_candles_ta.sort_values(by='timestamp_ms', ascending=True, inplace=True)
    return pd_candles_ta

def trade_files_2_dataframe(
    trade_files : List[str]
) -> pd.DataFrame:
    _trade_files = []
    for trade_file in trade_files:
        pd_flattened_trades = pd.read_csv(trade_file)
        pd_flattened_trades['entry_timestamp_ms'] = pd_flattened_trades['timestamp_ms'].shift(1) # Assume no overlapping trades
        pd_flattened_trades['entry_datetime'] = pd.to_datetime(pd_flattened_trades['entry_timestamp_ms'], unit='ms', utc=True).dt.tz_convert(TIMEZONE)
        pd_flattened_trades['entry_date'] = pd_flattened_trades['entry_datetime'].dt.date
        pd_flattened_trades['entry_dayofweek'] = pd_flattened_trades['entry_datetime'].dt.dayofweek
        pd_flattened_trades = pd_flattened_trades[(pd_flattened_trades['reason'] != 'entry') & (pd_flattened_trades['reason'] != 'HC')]
        _trade_files.append(pd_flattened_trades)

    pd_flattened_trades = pd.concat(_trade_files, ignore_index=True)
    return pd_flattened_trades

def filter_trades(pd_flattened_trades : pd.DataFrame) -> pd.DataFrame:
    # pd_flattened_trades = pd_flattened_trades[pd_flattened_trades.num_impacting_economic_calendars==0]
    # pd_flattened_trades = pd_flattened_trades[pd_flattened_trades.entry_dayofweek.isin([0,1,2,3,4])]
    # pd_flattened_trades = pd_flattened_trades[pd_flattened_trades.side=='buy'] # Include only long's (if exit side 'sell', entry side 'buy.)
    # pd_flattened_trades = pd_flattened_trades[~((pd_flattened_trades.entry_dayofweek==2) & (pd_flattened_trades.entry_hour.isin([9,10,11])))]
    return pd_flattened_trades

def calc_return(
        pd_flattened_trades : pd.DataFrame,
        return_calc_methodology : str = 'log'
    ):
    # pd_flattened_trades = pd_flattened_trades.loc[:,['trade_datetime', 'trade_year', 'trade_month', 'trade_day', 'trade_pnl_less_comm']]
    # pd_flattened_trades = pd_flattened_trades[pd_flattened_trades.timestamp_ms!=1740931200000]
    pd_flattened_trades.sort_values(by=['trade_datetime'], ascending=[True])
    pd_flattened_trades['trade_datetime'] = pd.to_datetime(pd_flattened_trades['trade_datetime'])
    pd_flattened_trades['total_equity'] = pd_flattened_trades['trade_pnl_less_comm'].cumsum()
    pd_flattened_trades['total_equity'] = pd_flattened_trades['total_equity'] + ACC_EQUITY

    if return_calc_methodology=='simple':
        pd_flattened_trades['interval_return'] = (pd_flattened_trades['total_equity'] / pd_flattened_trades['total_equity'].shift(1)) - 1 # Simple return
        pd_flattened_trades['interval_return'].fillna(0, inplace=True)
        pd_flattened_trades['cumulative_interval_return'] = (1 + pd_flattened_trades['interval_return']).cumprod() # cumprod is multiplicative, for Simple return

    elif return_calc_methodology=='log':
        pd_flattened_trades['interval_return'] = np.log(pd_flattened_trades['total_equity'] / pd_flattened_trades['total_equity'].shift(1)) # Log return
        pd_flattened_trades['interval_return'].fillna(0, inplace=True)
        pd_flattened_trades['cumulative_interval_return'] = pd_flattened_trades['interval_return'].cumsum() # cumsum is additive, for Log return

        # Check against final total_equity: This assumes you trades opens bigger and bigger (re-invest gains, assuming it's making money)
        pd_flattened_trades['growth_factor'] = np.exp(pd_flattened_trades['cumulative_interval_return'])
        pd_flattened_trades['final_equity_check'] = ACC_EQUITY * pd_flattened_trades['growth_factor'] # Will be different from final 'total_equity' if your strategy don't re-invest gains, under or over leverage!

    pd_flattened_trades['trade_datetime'] = pd.to_datetime(pd_flattened_trades['trade_datetime'])
    pd_flattened_trades.set_index('trade_datetime', inplace=True)

Load economic calendar

In [ ]:
pd_economic_calendars = pd.read_csv(ECONOMIC_CALENDAR_FILE)
pd_economic_calendars['datetime'] = pd.to_datetime(pd_economic_calendars['calendar_item_timestamp_ms'], unit='ms', utc=True).dt.tz_convert(TIMEZONE)
pd_economic_calendars['date'] = pd_economic_calendars['datetime'].dt.date
pd_economic_calendars = pd_economic_calendars[(pd_economic_calendars.importance==3) & (~pd_economic_calendars.event_code.isnull())]
economic_calendar_dates = pd_economic_calendars['date'].tolist()
economic_calendar_ts = pd_economic_calendars['calendar_item_timestamp_ms'].dropna().sort_values().to_numpy()
pd_economic_calendars

Load Candles TA

In [ ]:
if RELOAD_CANDLES:
    candles_ta_regex_filter : str = f"{TICKER}.*candles_ta.*_1h\.csv$"
    pattern = re.compile(candles_ta_regex_filter)
    cache_files = [os.path.join(CANDLES_CACHE_DIR, f) for f in os.listdir(CANDLES_CACHE_DIR) if pattern.match(f)]
    filtered_candles_ta_files = [f for f in cache_files if pattern.search(f)]

    print(f"Loading candles_ta ...")
    for candles_ta_file in filtered_candles_ta_files:
        print(f"{candles_ta_file}")
        
    pd_candles_ta = candles_ta_files_2_dataframe(candles_ta_files=filtered_candles_ta_files)

Custom TA calc

In [ ]:
CUSTOM_SLIDING_WINDOW_SiZE_ATR : int = 24 *90
CUSTOM_SLIDING_WINDOW_SiZE_ADX : int = 24 *5

pd_candles_ta.loc[:,'h_l'] = pd_candles_ta['high'] - pd_candles_ta['low']
pd_candles_ta.loc[:,'h_pc'] = abs(pd_candles_ta['high'] - pd_candles_ta['close'].shift(1))
pd_candles_ta.loc[:,'l_pc'] = abs(pd_candles_ta['low'] - pd_candles_ta['close'].shift(1))
pd_candles_ta.loc[:,'tr'] = pd_candles_ta[['h_l', 'h_pc', 'l_pc']].max(axis=1)
pd_candles_ta.loc[:,'atr'] = pd_candles_ta['tr'].rolling(window=CUSTOM_SLIDING_WINDOW_SiZE_ATR).mean()
pd_candles_ta.loc[:,'atr_bps'] = pd_candles_ta['atr']/pd_candles_ta['ema_close'] *10000

def wilder_smooth(series: pd.Series, period: int) -> pd.Series:
	if period <= 0:
		raise ValueError("period must be > 0")
	n = len(series)
	if n < period:
		# Not enough data; return NaNs
		return pd.Series(np.nan, index=series.index)
	
	alpha = 1.0 / period
	ewm = series.ewm(alpha=alpha, adjust=False).mean()
	pivot = period - 1
	init_avg = series.iloc[:period].mean()
	correction = init_avg - ewm.iloc[pivot]
	result = ewm.copy()
	indices = np.arange(pivot, n)
	decay = (1 - alpha) ** (indices - pivot)
	result.iloc[pivot:] = ewm.iloc[pivot:] + correction * decay
	result.iloc[:pivot] = np.nan
	return result
high, low = pd_candles_ta['high'], pd_candles_ta['low']
up_move = high - high.shift(1)
down_move = low.shift(1) - low

plus_DM = pd.Series(
	np.where((up_move > down_move) & (up_move > 0), up_move, 0.0),
	index=pd_candles_ta.index
)
minus_DM = pd.Series(
	np.where((down_move > up_move) & (down_move > 0), down_move, 0.0),
	index=pd_candles_ta.index
)
smoothed_plus_DM = wilder_smooth(plus_DM, CUSTOM_SLIDING_WINDOW_SiZE_ADX)
smoothed_minus_DM = wilder_smooth(minus_DM, CUSTOM_SLIDING_WINDOW_SiZE_ADX)
smoothed_atr = wilder_smooth(pd_candles_ta['tr'], CUSTOM_SLIDING_WINDOW_SiZE_ADX)
plus_DI = 100 * smoothed_plus_DM / smoothed_atr
minus_DI = 100 * smoothed_minus_DM / smoothed_atr
di_sum = plus_DI + minus_DI
di_diff = (plus_DI - minus_DI).abs()
dx = 100 * di_diff / di_sum
pd_candles_ta['adx'] = wilder_smooth(dx, CUSTOM_SLIDING_WINDOW_SiZE_ADX)

Load trade file

In [ ]:
pd_flattened_trades = trade_files_2_dataframe(trade_files=trade_files)
pd_flattened_trades = filter_trades(pd_flattened_trades=pd_flattened_trades)
calc_return(pd_flattened_trades=pd_flattened_trades, return_calc_methodology=RETURN_CALC_METHODOLOGY)
pd_flattened_trades

In [ ]:
simple_periods_pnl = pd_flattened_trades['trade_pnl_less_comm'].sum()
simple_periods_return = round(simple_periods_pnl/ ACC_EQUITY * 100, 2)
resampled_returns = pd_flattened_trades['interval_return'].resample(RETURN_RESAMPLE_FREQ).sum() # Resampled to Daily weekly ...etc.

metrics_series = qs.reports.metrics(
    returns=resampled_returns,
    rf=RISK_FREE_RATE,
    mode='full',
    display=False
).to_dict()
cumulative_return = round(metrics_series['Strategy']['Cumulative Return'] *100, 2)
sharpe_ratio = metrics_series['Strategy']['Sharpe']
max_drawdown = round(abs(metrics_series['Strategy']['Max Drawdown']) * 100, 2)
num_tp = pd_flattened_trades[pd_flattened_trades.reason.isin(['TP'])].shape[0]
num_trades = pd_flattened_trades[pd_flattened_trades.reason.isin(['TP', 'SL'])].shape[0]
hit_ratio = round(num_tp/num_trades, 2) * 100
duration_days = (pd_flattened_trades[pd_flattened_trades.reason.isin(['TP', 'SL'])].index.max() - pd_flattened_trades[pd_flattened_trades.reason.isin(['TP', 'SL'])].index.min()).days
num_trades_perday = num_trades/duration_days
num_trades_peryear = round(num_trades_perday*365, 2)
num_trades_permo = round(num_trades_peryear/12, 2) 
annualized_return_percent = round(cumulative_return/duration_days*365, 2)
mean_leverage_percent = round(pd_flattened_trades['current_position_usdt'].mean()/ACC_EQUITY*100, 2)

# Adjust to 1x leverage
adj_annualized_return_percent = round(annualized_return_percent / (mean_leverage_percent/100), 2) 
adj_max_drawdown = round(max_drawdown / (mean_leverage_percent/100), 2)

summary = f"simple_periods_return: {simple_periods_return}%, mean_leverage_percent: {mean_leverage_percent}%, hit_ratio: {hit_ratio}%, cumulative_return: {cumulative_return}% (Annualised {adj_annualized_return_percent}% adjusted to 1x), sharpe_ratio: {sharpe_ratio}, max_drawdown: {max_drawdown}% ({adj_max_drawdown}% adjusted to 1x), num_trades: {num_trades}, duration_days: {duration_days}, num_trades_permo: {num_trades_permo}"
print(summary)

Equity Curve

In [ ]:
pd_candles_ta = pd_candles_ta[(pd_candles_ta.timestamp_ms>=pd_flattened_trades['timestamp_ms'].min()) & (pd_candles_ta.timestamp_ms<=pd_flattened_trades['timestamp_ms'].max())]

fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(25, 10), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
plt.style.use('dark_background')

ax_top.set_title(summary, fontsize=12, color='white', pad=20)
ax_top.set_ylabel('Total Equity', color='white')

ax_top.plot(pd_flattened_trades.index, pd_flattened_trades['total_equity'],
            label='Total Equity', color='cyan')

sl_mask = pd_flattened_trades['reason'] == 'SL'
maxpain_mask = (np.abs(pd_flattened_trades['max_pain_percent']) > MAXPAIN_CUTOFF_BPS/100) & (pd_flattened_trades.reason == 'TP')

ax_top.scatter(pd_flattened_trades.index[sl_mask],
               pd_flattened_trades.loc[sl_mask, 'total_equity'],
               color='red', s=40, marker='v', label='SL')

ax_top.scatter(pd_flattened_trades.index[maxpain_mask],
               pd_flattened_trades.loc[maxpain_mask, 'total_equity'],
               color='orange', s=40, marker='^', label=f'Max Pain > {MAXPAIN_CUTOFF_BPS} bps')

ax2 = ax_top.twinx()
ax2.set_ylabel('Close Price', color='white')
ax2.grid(False)

candle_times = pd.to_datetime(pd_candles_ta['timestamp_ms'], unit='ms')
ax2.plot(candle_times, pd_candles_ta['close'],
         color='gray', linewidth=1.5, alpha=0.7, label='Close Price')
ax2.tick_params(axis='y', labelcolor='white')

lines1, labels1 = ax_top.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax_top.legend(lines1 + lines2, labels1 + labels2, loc='best')

ax_bottom.set_xlabel('Date', color='white')
ax_bottom.set_ylabel('adx', color='white')
ax_bottom.plot(candle_times, pd_candles_ta['adx'],
               color='magenta', linewidth=1.5, label='adx')
ax_bottom.legend(loc='best')

threshold = pd_candles_ta['adx'] > 20
adx_below_shifted = threshold.shift(1).fillna(False)
start = threshold & ~adx_below_shifted
end = ~threshold & adx_below_shifted
start.iloc[0] = threshold.iloc[0]
end.iloc[-1] = threshold.iloc[-1]
starts = candle_times[start]
ends = candle_times[end]
for st, en in zip(starts, ends):
    ax_top.axvspan(st, en, alpha=0.2, color='yellow')
    
plt.show()

Pnl contribution: Longs vs Shorts

In [ ]:
tp_data = pd_flattened_trades[pd_flattened_trades['reason'] == 'TP']
tp_pnl = tp_data.groupby('side')['trade_pnl_less_comm'].sum()

sl_data = pd_flattened_trades[pd_flattened_trades['reason'] == 'SL']
sl_pnl_abs = sl_data.groupby('side')['trade_pnl_less_comm'].apply(lambda x: abs(x).sum())

side_labels = {'sell': 'Long', 'buy': 'Short'} # if exit side 'sell', entry side 'buy

plt.style.use('dark_background')  # built‑in dark theme

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#1e1e1e')

# 1. TP contribution
colors = ["#FF0000", '#4CAF50']  # green for long, red for short
wedges1, texts1, autotexts1 = ax1.pie(
    tp_pnl,
    labels=tp_pnl.index.map(side_labels),
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    textprops={'color': 'white', 'fontsize': 12},
    wedgeprops={'edgecolor': 'none'}
)
ax1.set_title('TP', color='white', fontsize=14, pad=15)

# 2. SL contribution
wedges2, texts2, autotexts2 = ax2.pie(
    sl_pnl_abs,
    labels=sl_pnl_abs.index.map(side_labels),
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    textprops={'color': 'white', 'fontsize': 12},
    wedgeprops={'edgecolor': 'none'}
)
ax2.set_title('SL', color='white', fontsize=14, pad=15)

# Improve visibility of percentage text
for autotext in autotexts1 + autotexts2:
    autotext.set_color('white')
    autotext.set_fontsize(11)
    autotext.set_weight('bold')

plt.tight_layout()
plt.show()

In [ ]:
pd_flattened_trades['max_pain_bps'] = round(pd_flattened_trades['max_pain_percent']*100, 2)

plt.figure(figsize=(12, 6), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

plt.hist(
    pd_flattened_trades[(pd_flattened_trades.reason=='TP')]['max_pain_bps'],
    bins=25,
    edgecolor='#444444',
    alpha=0.85,
    color='dodgerblue'
)

mean_max_pain_bps = round(pd_flattened_trades[pd_flattened_trades.reason=='TP']['max_pain_bps'].mean(), 2)
plt.axvline(mean_max_pain_bps, color='limegreen', linestyle='-', linewidth=1.3, alpha=0.85,
            label=f'Mean max_pain_bps {mean_max_pain_bps}')

plt.title('max_pain_bps distribution  (TP only)', 
          fontsize=14, fontweight='bold', color='white')
plt.xlabel('max_pain_bps', fontsize=12, color='white')
plt.ylabel('# trades', fontsize=12, color='white')

plt.grid(True, alpha=0.15, linestyle='--', color='gray')
plt.legend(loc='upper right', frameon=False, fontsize=10.5, labelcolor='white')

ax.tick_params(colors='white', which='both')
ax.spines['bottom'].set_color('gray')
ax.spines['top'].set_color('gray')
ax.spines['left'].set_color('gray')
ax.spines['right'].set_color('gray')

plt.tight_layout()
plt.show()

In [ ]:
pd_flattened_trades['trade_pnl_less_comm_bps'] = round(pd_flattened_trades['trade_pnl_less_comm']/pd_flattened_trades['current_position_usdt']*10000, 2)

plt.figure(figsize=(12, 6), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

plt.hist(
    pd_flattened_trades[pd_flattened_trades.reason=='TP']['trade_pnl_less_comm_bps'],
    bins=25,
    edgecolor='#444444',
    alpha=0.85,
    color='dodgerblue'
)

mean_trade_pnl_less_comm_bps = round(pd_flattened_trades[pd_flattened_trades.reason=='TP']['trade_pnl_less_comm_bps'].mean(), 2)
plt.axvline(mean_trade_pnl_less_comm_bps, color='limegreen', linestyle='-', linewidth=1.3, alpha=0.85,
            label=f'Mean trade_pnl_less_comm_bps {mean_trade_pnl_less_comm_bps}')

plt.title('trade_pnl_less_comm_bps distribution (TP only)', 
          fontsize=14, fontweight='bold', color='white')
plt.xlabel('trade_pnl_less_comm_bps', fontsize=12, color='white')
plt.ylabel('# trades', fontsize=12, color='white')

plt.grid(True, alpha=0.15, linestyle='--', color='gray')
plt.legend(loc='upper right', frameon=False, fontsize=10.5, labelcolor='white')

ax.tick_params(colors='white', which='both')
ax.spines['bottom'].set_color('gray')
ax.spines['top'].set_color('gray')
ax.spines['left'].set_color('gray')
ax.spines['right'].set_color('gray')

plt.tight_layout()
plt.show()

In [ ]:
net_by_day = pd_flattened_trades.groupby('entry_dayofweek')['trade_pnl_less_comm'].sum()

day_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
labels = [day_names[d] for d in net_by_day.index]
values = net_by_day.values

abs_values = np.abs(values)
colors = ['#4CAF50' if v >= 0 else '#FF5252' for v in values]

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(8, 8))
fig.patch.set_facecolor('#1e1e1e')

wedges, texts, autotexts = ax.pie(
    abs_values,
    labels=labels,
    autopct=lambda pct: f'{pct:.1f}%',
    startangle=90,
    colors=colors,
    textprops={'color': 'white', 'fontsize': 12},
    wedgeprops={'edgecolor': 'none'}
)

ax.set_title('Net PnL by Entry Day of Week (TP - SL)', color='white', fontsize=14, pad=15)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(11)
    autotext.set_weight('bold')

plt.tight_layout()
plt.show()

In [ ]:
grouped = pd_flattened_trades[pd_flattened_trades.reason.isin(['TP','SL'])].groupby(['entry_hour', 'entry_dayofweek']).agg(
    sum_trade_pnl_less_comm_bps=('trade_pnl_less_comm_bps', 'sum'),
).reset_index()

min_sum_trade_pnl_less_comm_bps = grouped['sum_trade_pnl_less_comm_bps'].min()
max_sum_trade_pnl_less_comm_bps = grouped['sum_trade_pnl_less_comm_bps'].max()

heatmap_data = grouped.pivot(index='entry_hour', columns='entry_dayofweek', values='sum_trade_pnl_less_comm_bps')

plt.figure(figsize=(15, 7), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

sns.heatmap(
    heatmap_data, 
    annot=True, 
    fmt='.2f', 
    cmap='RdYlGn', 
    vmin=min_sum_trade_pnl_less_comm_bps,
    vmax=max_sum_trade_pnl_less_comm_bps,
    center=0,
    cbar_kws={'label': 'avg_macd_length by hour and day of week'}
)

ax.set_title('trade_pnl_less_comm_bps by hour and day of week', color='white')
ax.set_xlabel('day of week', color='white')
ax.set_ylabel('hour', color='white')
ax.tick_params(colors='white')
cbar = ax.collections[0].colorbar
cbar.set_label('avg_macd_length', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

plt.show()

In [ ]:
grouped = pd_flattened_trades[pd_flattened_trades.reason.isin(['TP','SL'])].groupby(['entry_hour', 'entry_dayofweek']).agg(
    mean_max_pain=('max_pain', 'mean'),
).reset_index()

min_mean_max_pain = grouped['mean_max_pain'].min()
max_mean_max_pain= grouped['mean_max_pain'].max()

heatmap_data = grouped.pivot(index='entry_hour', columns='entry_dayofweek', values='mean_max_pain')

plt.figure(figsize=(15, 7), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

sns.heatmap(
    heatmap_data, 
    annot=True, 
    fmt='.2f', 
    cmap='RdYlGn', 
    vmin=min_mean_max_pain,
    vmax=max_mean_max_pain,
    center=0,
    cbar_kws={'label': 'max_pain by hour and day of week'}
)

ax.set_title('max_pain by hour and day of week', color='white')
ax.set_xlabel('day of week', color='white')
ax.set_ylabel('hour', color='white')
ax.tick_params(colors='white')
cbar = ax.collections[0].colorbar
cbar.set_label('avg_macd_length', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

plt.show()

In [ ]:
qs.reports.full(
    returns=resampled_returns, 
    rf=RISK_FREE_RATE,
    title="Tear Sheet"
    )